# DADA-2000 → 221 test clips**This notebook moves files. It does not run a model, does not need a GPU, and produces no numbers.**Every measurement in this project stays on the Mac, so nothing here can affect a result.**Why it exists:** DADA-2000's only Drive mirror is a 6-part split zip totalling ~117 GB. The Mac has34 GB free. Colab has ~107 GB. Neither can hold it. But we need only the **221 clips** in`dada2000_small_test_concensus.csv` (~6 GB), and `7z` can pull named members out of a split zip**without ever unpacking the whole thing** — it reads through the volumes off the mounted Drive andwrites only what we ask for.**Runtime: pick CPU (no GPU).** A GPU runtime gets *less* disk and we have no use for one.**Before running:** in Drive, open the shared DADA2000 folder (the one containing `DADA2000.z01` …`DADA2000.zip`), right-click → Organise → **Add shortcut** → **My Drive**. A shortcut costs **0** ofyour storage quota — it is a link, not a copy.⚠️ **Do not make your copy of this folder shared or public.** DADA-2000 posts no licence. It is forinternal falsification only.**Order of cells matters.** Cells 2–4 are cheap and exist to catch a surprise *before* the expensivestep. If cell 4 reports the archive holds image frames rather than `.mp4` files, **stop** — ourscoring path decodes video, and that would be a different job.

## 1 — Mount Drive and find the six parts

In [40]:
from google.colab import drive
drive.mount("/content/drive")

import glob, os

# The shortcut may land at slightly different paths depending on where it was added.
CANDIDATES = [
    "/content/drive/MyDrive/DADA2000",
    "/content/drive/MyDrive/DADA2000/DADA2000",
]
SRC = next((p for p in CANDIDATES if glob.glob(os.path.join(p, "DADA2000.z*"))), None)

if SRC is None:
    hits = glob.glob("/content/drive/MyDrive/**/DADA2000.zip", recursive=True)
    SRC = os.path.dirname(hits[0]) if hits else None

assert SRC, ("Could not find DADA2000.zip under MyDrive. Add the shortcut to My Drive, "
             "then Runtime > Restart and run again.")

parts = sorted(glob.glob(os.path.join(SRC, "DADA2000.z*")))
total = sum(os.path.getsize(p) for p in parts)
for p in parts:
    print(f"{os.path.basename(p):18s} {os.path.getsize(p)/2**30:7.2f} GiB")
print(f"\n{len(parts)} parts, {total/2**30:.2f} GiB total")
print("SRC =", SRC)

# The .zip is the LAST volume and holds the central directory; 7z must be pointed at it.
LAST = os.path.join(SRC, "DADA2000.zip")
assert os.path.exists(LAST), "DADA2000.zip (the final volume) is missing -- 7z needs it to read the index"
assert len(parts) == 6, f"expected 6 parts, found {len(parts)} -- do not proceed with a partial set"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DADA2000.z01         19.90 GiB
DADA2000.z02         19.90 GiB
DADA2000.z03         19.90 GiB
DADA2000.z04         19.90 GiB
DADA2000.z05         19.90 GiB
DADA2000.zip         17.25 GiB

6 parts, 116.75 GiB total
SRC = /content/drive/MyDrive/DADA2000


## 2 — Read the archive index`7z l` reads the central directory from the last volume. This is the cheap look inside: it tells uswhat the archive actually holds **before** we spend time extracting anything.

In [41]:
!apt-get -qq install -y p7zip-full > /dev/null 2>&1
import subprocess, time

t0 = time.time()
listing = subprocess.run(["7z", "l", "-ba", LAST], capture_output=True, text=True, timeout=7200)
print(f"7z l took {time.time()-t0:.0f}s, exit {listing.returncode}")

if listing.returncode != 0:
    print("STDERR:\n", listing.stderr[-3000:])
    raise SystemExit("7z could not read the archive -- see stderr above. Do not continue.")

lines = [l for l in listing.stdout.splitlines() if l.strip()]
print(f"{len(lines)} entries\n")
print("\n".join(lines[:15]))
with open("/content/listing.txt", "w") as f:
    f.write(listing.stdout)

7z l took 31s, exit 0
3909940 entries

2022-11-09 00:21:22 D....            0            0  DADA2000
2022-10-29 07:52:57 D....            0            0  DADA2000/1
2022-09-23 07:22:48 D....            0            0  DADA2000/1/001
2022-10-28 12:28:19 D....            0            0  DADA2000/1/001/fixation
2020-05-22 01:25:50 ....A         1239          293  DADA2000/1/001/fixation/0001.png
2020-05-22 01:25:50 ....A         1239          293  DADA2000/1/001/fixation/0002.png
2020-05-22 01:25:50 ....A         1239          293  DADA2000/1/001/fixation/0003.png
2020-05-22 01:25:50 ....A         1249          305  DADA2000/1/001/fixation/0004.png
2020-05-22 01:25:50 ....A         1250          311  DADA2000/1/001/fixation/0005.png
2020-05-22 01:25:50 ....A         1250          311  DADA2000/1/001/fixation/0006.png
2020-05-22 01:25:50 ....A         1250          311  DADA2000/1/001/fixation/0007.png
2020-05-22 01:25:50 ....A         1250          311  DADA2000/1/001/fixation/0008.png
20

In [42]:
# === DIAGNOSTIC: structure + frame rate. No extraction — reads /content/listing.txt ===
import base64, collections, gzip, json, os

# The 221 test clips and their annotated Time-of-collision, embedded (gzip+b64) straight
# from vendor/badas-open/annotation/dada2000_small_test_concensus.csv so this needs no
# network and cannot drift from the repo copy.
ANN = json.loads(gzip.decompress(base64.b64decode(
"H4sIAHK1q2oC/32Yy45YNQyG32XW5SiJ7cRhy54XQNUIIRaVKirBEvHuJLbP6XTye9ZfnIvvzr8vtbyWri8//zauRkSfXn759vXrl3++fPvr5fMnp3NRvUrvA9BRFp2XJLQu2i5GqC1El+BDBy1ar4kQ232aYkGxtxSEup2oCI2FevpGNUF4ma2cSlfHkrq1IxdEWzV8USLY7P3orkr2wooVp2yCSK31tcxh9oDInUATwem0nretr7UUoxBVuw/Sz6bNlODe8+ufv//90x/HErIrK5Jms9pM9hbbmxPazYsIoWGCFWqiFtcTfXTl6Y6GN6geMYyQKQvap9b25tyTupYSVVT+6EVVPMAR6ua/U7GgR012JU0NV6cJMr5P86BBLtxeS3WzimLaXQ+J7DAdIpeg12LuNGAS4CVrGaleVRC0qGP4In7ujM6VdW51hzg27sFwlhhBO9SUxnsajNkZsktTx6mvpTW3OkJbCwXeZ0FX4MhCY60QT2toZ8/ODOy64LAbeYU6d24l3rMufb621UePRZMNlnOUkQWYUbUg2ToB4vSc38H5HHR5T3vPbvvrVcHB/bETg323B/T00iOeVOGdln+QG6PBF23uwU9INE2RRtUDuGI/sCXTfVOyBezPznzJlniNnB8cw2RO1TOzrzho03Pu+zy1GZWs6hm9bdoxbv6CRJjcowUxtoRAkInv+h5FuMtVTptQiQy1krwgJiZZFUumNYDq7WIlDHVi9fYQCN8h15Ab0R1yFSYwWgHnxbJEI3ga19Z4f5FzMq0IOGEFraqfAPKrYS/zCE0vYQ0hj3ZQqY02b0QFMXfmhm8z3fyUUMn6A6PdPZkUC3udr4mwN0Q9od44EtSwN461JsLRPK4nJ9Jepxg+2RtHSU/msC1iEupArHspRyi8GSH1hid55vTSAbS/y4PAGYp6ZH+QKGg8FQ90QqSRYhr0lrsyYNXddQEb5S4NenWY2emuDgy96S4MK+gr3p3Lh9033ZUDDjK0033xlgkh78NAB2G02bsGlPSRTBHyvHtYL4wn8KGRGem0LL8ZKiHzcQvEOd9TJSeSlJV0/j5WlmRfgdO6oe4JrWJBD5mGkFcOSgQnnI83stxbr4EfMr1lLgltMNYM+Y+EJoLsmSpRkKXfnLqOOkLewWkiqPl1p4f/YeqnZjfUNfL3mv0+j3F7RqTjprc7E2oi+PZnOYcn3r1x81TcUUSzxAJC2ZP7M/Ihe/cY+dbmjJi+6TFP6kPqGYIjsggsp0a9d20J9deANGOUsuxlNH48ANXQU0PdDM8wK8fOQNF7yfQxlpKWmp8pCd3g7tgIBa2U536ggZEahw/k6pvar8n9z3beTNozKWVjh9BTFIGGhMNXCBVFo+4toFUx6rcveGfzF0a/pMu9O2d/FhtKlj837O7cRaHo8DjGh2qWkSS+fCdURPz4wi8JiQ/fsxJI/PdqJuZeL4lvSvz6EpqmJD59Cc3IEt++zRqSA7nBCyA+wiY7zjeR9ANcaYO9mZsnEfwZvElMkgPuKEnV3ojMOxognH05bej/UQ0f17OSs6EP/qD32zC6XAxnVpYX7CU78/7jqLBj1Ei28ANeI9c25OUaqVZQ3GlkWokMf3qkxl8WmODnk8rAhWdkMjm957//ARWizMaNGQAA"
)))
assert len(ANN) == 221, len(ANN)
tcoll = {k: v[0] for k, v in ANN.items()}

paths = []
for l in open("/content/listing.txt"):
    f = l.split(None, 5)
    if len(f) == 6:
        paths.append(f[5].replace("\\", "/").strip())
print(f"{len(paths)} entries\n")

# 1. sub-folders per clip:  DADA2000/{type}/{clip}/{sub}/NNNN.png
subs = collections.Counter()
for p in paths:
    q = p.split("/")
    if len(q) >= 5 and q[0] == "DADA2000":
        subs[q[3]] += 1
print("per-clip sub-folders:", subs.most_common(12), "\n")

# 2. frame counts per clip per sub-folder
counts = collections.defaultdict(collections.Counter)
for p in paths:
    q = p.split("/")
    if len(q) == 5 and q[0] == "DADA2000" and q[4].lower().endswith(".png"):
        counts[f"{q[1]}_{q[2]}"][q[3]] += 1
print(f"clips in archive: {len(counts)}")
print(f"example {sorted(counts)[0]}: {dict(counts[sorted(counts)[0]])}\n")

present = [k for k in tcoll if k in counts]
print(f"wanted 221   present in archive {len(present)}")
if len(present) < 221:
    print("  missing:", [k for k in tcoll if k not in counts][:10])

# 3. THE FRAME-RATE TEST.
#    t = index/fps, so duration = n_frames/fps. Every annotated collision must land
#    INSIDE its own clip. Too high an fps shortens the clip and pushes collisions past
#    the end -> falsified. Too low only stretches it, and can never be falsified this
#    way. So this gives an UPPER BOUND: the HIGHEST consistent rate is the estimate,
#    and at the true rate the tightest margin should be small and positive.
print("\n" + "=" * 64)
print("FRAME RATE — highest rate consistent with every annotation")
print("=" * 64)
for sub in subs:
    n = {k: counts[k][sub] for k in present if counts[k].get(sub)}
    if len(n) < 0.9 * len(present):
        continue
    print(f"\n  '{sub}'  ({len(n)} clips, frames {min(n.values())}-{max(n.values())})")
    ok = []
    for fps in (10, 15, 20, 24, 25, 30, 60):
        over = [k for k in n if tcoll[k] * fps > n[k]]
        slack = min((n[k] / fps) - tcoll[k] for k in n)
        print(f"     {fps:3d} fps  {'OK' if not over else f'X {len(over)} past end':16s}"
              f"  tightest margin {slack:+7.2f}s")
        if not over:
            ok.append(fps)
    print(f"     -> highest consistent: {max(ok) if ok else 'NONE'}")

# 4. extraction cost
print()
for sub in subs:
    tot = sum(counts[k].get(sub, 0) for k in present)
    if tot:
        print(f"  '{sub}' for {len(present)} clips = {tot:,} png files")


3909940 entries

per-clip sub-folders: [('fixation', 1300678), ('maps', 649363), ('images', 649358), ('seg', 649358), ('semantic', 649358)] 

clips in archive: 1962
example 10_001: {'fixation': 878, 'images': 878, 'maps': 878, 'seg': 878, 'semantic': 878}

wanted 221   present in archive 132
  missing: ['11_098', '22_007', '22_008', '25_001', '26_001', '27_006', '27_007', '28_035', '28_036', '28_037']

FRAME RATE — highest rate consistent with every annotation

  'fixation'  (132 clips, frames 84-868)
      10 fps  X 1 past end      tightest margin   -3.67s
      15 fps  X 1 past end      tightest margin   -6.57s
      20 fps  X 3 past end      tightest margin   -8.02s
      24 fps  X 5 past end      tightest margin   -8.74s
      25 fps  X 6 past end      tightest margin   -8.89s
      30 fps  X 11 past end     tightest margin   -9.47s
      60 fps  X 78 past end     tightest margin  -10.92s
     -> highest consistent: NONE

  'images'  (132 clips, frames 84-868)
      10 fps  X 1 pas

In [43]:
# === DIAGNOSTIC 2: why do the annotation and the archive disagree? ===
import base64, collections, gzip, json, os, glob

ANN = json.loads(gzip.decompress(base64.b64decode(
"H4sIANq/q2oC/31ZS44FNQy8y6wfrSSO82HLngsgNEKIBRICCZaIuxO73N2P6QqrWXRN4vhTLvv9/ZHTZ2rj49sf+lFE5KVHzq2/Pr7747fffv3r1z9+//jxBdBcoHGktr62o4yFfYJ6WqB5KEB9A8oLVI76ygf9XNZnOdSsyUcSfoYsUD7mK21vqW6vG9qO0eijuvrL03p2Vfa9uS1jGVI3J/SFaHBLPTI9ZPgh81WOyX3bzbdZjman5HQIf9Iw7+phFzVuzTDX1kPs6/a2Udx3Y/lOuHeHuFvy/ybEqH6Mxykd7Ssif6bZPdAWxkpeZAhk3hAPkpCLDDQByp5U84nIKTmiL2P5GTllN9bdlo6iDFLcdwrf+Z+P73/56c9vfn4gxd/lWSHsXTlVz4pph8hBDVK/reI2blDzDHbP0O/dj/AolSNtHg4Py3vZbl41kes4jiXYAmUUeH2tomE2ZfezxzzR/DNMuU2qR+cezPCxezDT8jVQffdAGfwkBXGtQAxqcvOqmvbfi3YYAjXuBjOiMMi4U2Jj6/RDPGFW/XJbCyrcaypC8QVUPlNG6igMpogG58EveXNM9zDU4OrHq+QzeRp3kJuSJK3rFCfibMw3aJUbxpmi4u0rOQrD/KcelPhQlz0Z2afG6A9rWgCC/1bdkVN6gBqcrCsvviBGvLuAcdKzfGecsTys5l/y6s9UCtJKX2HPE2GuS/iYaJQWBhHwl3RvL6R4F0xB6WpMzM1BJ/MI8c6wMN1NLkhReldJ8XbFGZX1qpKvSCTclzZsWlYepv7OUPo1rA4ZXsLLl9Er2FFyWdZgGQl/qQFaGVuMotrjujPNxpGR9iS8pV0ZULMT1HxGr1iqtfttytLAQR29CSc9qqyshBTEt1jLpo83DHjMQlee4XXEfxsGMdlAAxSUvYTGpmE4cqJK1KyWPa7CUx1UXnWLgy6Z8MMeJp7JLTJ5Z+Aq0jLRjIaRDqkvw0i6xUVnTdRBZ8rYEyZTnI4qeCfsl81RghozWpgbk6qznFi2K/uuuMfSjmRU0JciwPLUpJKCkof1cNqLHKJ+Sh47PeCgt+ZJU0rymeAJGTC3qIHZwY56PlxOpihn/maGGBFxjwHVDbJ4AiolQTHollIcClVoF6puQeLe9JpRRq6yeGcMXOp9ZyKXCAoqzDTzo8PZ9wmFUF7Cuq0jwF4TnNP5NbNgbNmkoUNQZyXYYnMZUk2QanVzmd5ajieJgRpqzPvJctPuQmixDPEo1EWQvC0cwBCYKAQTRacGYajIMTNMUkb1HCsEhLuwlWEgBGrIDXoMSEHy9lGYKCx7limVJhhGCvPgUmI8uzBSLJVl4mjz6qgxYzBqx4CMhf/T5pCJPjxcSHRirnVZvdYAjIFadE8nwvHUp9IvkTEhWB5HjODSgvwk4Tlba/heSHjOznoGOVriE9S97Jqxdt80JTn7a0UCM3I+G6u1rIx1AAumNcL0LpzS7kJvv6GbKiUNa5UJmniZRHqKI6C9JxqlbEDFnbCCFQqCQLBbGK9G6WKG5l3DxGuQcERWxFaCWxudQixz8mNhUN82KUZ/RBjUc5WSzwGUqNx6rlKqnWMqg3yXW3GRYNd7jZIwBcjGGL3WXaSlOaKB0EO2bY5BcReTV5sHoQl7NQwmhxw0Y4FEh11DeJfK4JityRNzWMiUHajcLLFBYBU4cNdkPvYutWjanWyygGHUUwaQzTHwcVupm7n7JqT8QDD7BjSuJym9ZoLVvBhJIC9FVVCGiyEGCdQtqfImlOWa8O1JpF7rWW5yyUBlEMyN2arp62xfbeIq6F1t2aEbrVU1cLLdidV2bTZ6SEAKCbVp/EFoyiHjfSIh/clBWNiIcdlXrVl7UOa1oeoMgcGnAMF4pgdlCnhVqC39Jk1xbZw3oFg6bre2dYSPC4LNS2pG3lRcJx4IEjDDTSx2nC5WfeXNmF+vKX9i01uIL071LnFcZmOspusJJXr68yjNYVpHJSrroAbybWas3YVP1lquUT98JpvmrnIJEoWQY6bVSFAB1ZctCCnatkziIDwyYe4gDGogz9PYawuTWqvyWr0XhvlZ5IbQu79UxhWGaag8l+6DBdlAHZyEjdbGnPHOxQww0aZQD6TKNX6LunaB5Ek9378hUVM71IzE8o0AUJaxSaSpr/FLVJQ23VJr/BAlUPZ0l6/xY5RTcGY6QePHqOU3vn7Q+ClKcU3dBKjP9+JvT8zixgqBP7nUWAA9f6oSlrsGiJVKjxBy0CWv6mNPa5/Fc7Ksr6QBGqDeS2Vhw4phsHgueG7i57S7rZO5yBBYrRXET/hNMTpFCTV603wXTuSilt4VBnH+uXXMMWMslfCYnUb0pvMXx9iiPTAg2pCmhF5GdCY9R7DMbsKKYGJi3P0CNWKJ7asxeRTbvBg9Jn99vnwGoeuZuw+R9s+/8EVhaPAeAAA="
)))
tcoll = {k: v[0] for k, v in ANN.items()}
talert = {k: v[1] for k, v in ANN.items()}
print(f"{len(ANN)} annotated clips\n")

paths = []
for l in open("/content/listing.txt"):
    f = l.split(None, 5)
    if len(f) == 6:
        paths.append(f[5].replace("\\", "/").strip())

counts = collections.defaultdict(int)
for p in paths:
    q = p.split("/")
    if len(q) == 5 and q[0] == "DADA2000" and q[3] == "images" and q[4].lower().endswith(".png"):
        counts[f"{q[1]}_{q[2]}"] += 1

# ---- A. how are clips numbered per type folder? Is "11_098" even plausible?
per_type = collections.defaultdict(list)
for k in counts:
    t, c = k.split("_")
    per_type[t].append(int(c))
print("A. clips per type folder (archive):")
for t in sorted(per_type, key=lambda x: int(x))[:8]:
    v = sorted(per_type[t])
    print(f"   type {t:>3s}: {len(v):3d} clips, numbered {v[0]}..{v[-1]}")
print(f"   ... {len(per_type)} type folders, {len(counts)} clips total\n")

want_type = collections.Counter(k.split('_')[0] for k in tcoll)
print("B. what we need vs what exists:")
for t in sorted(want_type, key=lambda x: int(x))[:10]:
    have = sorted(per_type.get(t, []))
    need = sorted(int(k.split('_')[1]) for k in tcoll if k.split('_')[0] == t)
    print(f"   type {t:>3s}: need {need[:6]}{'...' if len(need)>6 else ''} "
          f"| archive has {len(have)} numbered {have[0] if have else '-'}..{have[-1] if have else '-'}")

# ---- C. the impossible ones, with numbers
print("\nC. worst mismatches (matched clips only), at 30 fps:")
bad = []
for k in tcoll:
    if k in counts:
        dur = counts[k] / 30.0
        bad.append((tcoll[k] - dur, k, counts[k], dur, tcoll[k], talert[k]))
bad.sort(reverse=True)
print(f"   {'clip':>10s} {'frames':>7s} {'dur@30':>8s} {'t_coll':>8s} {'t_alert':>8s} {'over by':>8s}")
for d, k, n, dur, tc, ta in bad[:10]:
    print(f"   {k:>10s} {n:7d} {dur:8.2f} {tc:8.2f} {ta:8.2f} {d:+8.2f}")
n_over = sum(1 for d, *_ in bad if d > 0)
print(f"\n   {n_over}/{len(bad)} matched clips have t_coll AFTER the clip ends at 30 fps")

# ---- D. if instead the annotation were in FRAMES not seconds, does it fit?
fits = sum(1 for k in tcoll if k in counts and tcoll[k] <= counts[k])
print(f"\nD. if t_coll were a FRAME index: {fits}/{len(bad)} would fit inside the clip")

# ---- E. the official DADA spreadsheet sitting next to the archive
xl = glob.glob(os.path.join(SRC, "*.xlsx"))
print(f"\nE. spreadsheet: {xl}")
if xl:
    import pandas as pd
    for sheet, df in pd.read_excel(xl[0], sheet_name=None).items():
        print(f"\n   sheet '{sheet}': {df.shape[0]} rows x {df.shape[1]} cols")
        print("   columns:", list(df.columns)[:15])
        print(df.head(4).to_string()[:1200])


221 annotated clips

A. clips per type folder (archive):
   type   1:  53 clips, numbered 1..53
   type   2:   6 clips, numbered 1..6
   type   3:  17 clips, numbered 1..17
   type   4:  13 clips, numbered 1..13
   type   5: 157 clips, numbered 1..157
   type   6: 119 clips, numbered 1..119
   type   7:  12 clips, numbered 1..12
   type   8:  57 clips, numbered 1..57
   ... 52 type folders, 1962 clips total

B. what we need vs what exists:
   type   1: need [22, 23, 24, 25, 26, 27] | archive has 53 numbered 1..53
   type   2: need [4] | archive has 6 numbered 1..6
   type   3: need [6, 7] | archive has 17 numbered 1..17
   type   4: need [7, 8] | archive has 13 numbered 1..13
   type   5: need [64, 65, 66, 67, 68, 69]... | archive has 157 numbered 1..157
   type   6: need [49, 50, 51, 52, 53, 54]... | archive has 119 numbered 1..119
   type   7: need [6] | archive has 12 numbered 1..12
   type   8: need [21, 22, 23, 24, 25] | archive has 57 numbered 1..57
   type   9: need [9, 10] | ar

In [44]:
# === DIAGNOSTIC 3: can the spreadsheet replace the BADAS annotation? ===
import collections, glob, os
import pandas as pd, numpy as np

df = pd.read_excel(glob.glob(os.path.join(SRC, "*.xlsx"))[0], sheet_name="Sheet1")
df.columns = [str(c) for c in df.columns]
V, T = "video", "type"
ACC = [c for c in df.columns if "accident frame" in c][0]
TOT = [c for c in df.columns if "total frames" in c][0]
OCC = [c for c in df.columns if "accident occurred" in c][0]
print(f"{len(df)} rows   accident-occurred: {df[OCC].value_counts().to_dict()}\n")

# archive: frames per clip
counts = collections.defaultdict(int)
for l in open("/content/listing.txt"):
    f = l.split(None, 5)
    if len(f) != 6:
        continue
    q = f[5].replace("\\", "/").strip().split("/")
    if len(q) == 5 and q[0] == "DADA2000" and q[3] == "images" and q[4].lower().endswith(".png"):
        counts[(int(q[1]), int(q[2]))] += 1
print(f"archive: {len(counts)} clips with images/\n")

# ---- A. is `video` an index WITHIN its type, or global 1..1962?
print("A. sheet rows per type vs archive clips per type:")
sheet_per_type = df.groupby(T)[V].agg(["count", "min", "max"])
arch_per_type = collections.Counter(t for t, c in counts)
agree = 0
for t in sorted(arch_per_type)[:8]:
    s = sheet_per_type.loc[t] if t in sheet_per_type.index else None
    print(f"   type {t:>3d}: sheet {int(s['count']) if s is not None else '-':>4} rows "
          f"(video {int(s['min']) if s is not None else '-'}..{int(s['max']) if s is not None else '-'})"
          f"   archive {arch_per_type[t]:>4} clips")
for t in arch_per_type:
    if t in sheet_per_type.index and int(sheet_per_type.loc[t, "count"]) == arch_per_type[t]:
        agree += 1
print(f"   -> counts agree for {agree}/{len(arch_per_type)} type folders\n")

# ---- B. 🔴 THE DECISIVE CHECK: does sheet 'total frames' equal the PNGs on disk?
#        If yes, the (type, video) -> folder mapping is proven, not assumed.
print("B. sheet 'total frames' vs actual PNG count per clip:")
matched, exact, off = 0, 0, []
for _, r in df.iterrows():
    key = (int(r[T]), int(r[V]))
    if key in counts:
        matched += 1
        d = counts[key] - int(r[TOT])
        if d == 0:
            exact += 1
        else:
            off.append((abs(d), key, counts[key], int(r[TOT])))
print(f"   mapped {matched}/{len(df)} rows to a folder")
print(f"   EXACT frame-count match: {exact}/{matched}")
if off:
    off.sort(reverse=True)
    print("   worst disagreements (type,video): archive_png vs sheet_total")
    for d, k, a, s in off[:6]:
        print(f"      {k}  {a} vs {s}   diff {a-s:+d}")
if matched and exact / matched > 0.95:
    print("   ✅ mapping CONFIRMED by an independent quantity")
else:
    print("   ❌ mapping NOT confirmed — do not proceed on this")

# ---- C. the quantity gate 3a actually needs: where is the collision in the clip?
pos = []
for _, r in df.iterrows():
    key = (int(r[T]), int(r[V]))
    if key in counts and r[OCC] == 1 and r[TOT] and r[ACC] and 0 < r[ACC] <= r[TOT]:
        pos.append((key, int(r[ACC]), int(r[TOT]), r[ACC] / r[TOT]))
p = np.array([x[3] for x in pos])
sec = np.array([x[1] / 30.0 for x in pos])
print(f"\nC. collision position over {len(pos)} accident clips (fps assumed 30 for seconds):")
print(f"   normalised  median {np.median(p):.3f}   IQR {np.percentile(p,75)-np.percentile(p,25):.3f}"
      f"   range {p.min():.3f}-{p.max():.3f}")
print(f"   seconds     median {np.median(sec):.2f}s  IQR {np.percentile(sec,75)-np.percentile(sec,25):.2f}s"
      f"  range {sec.min():.2f}-{sec.max():.2f}s")
print(f"   in final 10% of clip: {100*(p>0.9).mean():.1f}%      (Nexar positives: 73.7%)")
print(f"\n   compare: BADAS csv IQR 4.63s | DAD 0.16s (degenerate) | floor for gate 3a 1.0s")
print(f"   mid-clip collisions (0.2<pos<0.8): {100*((p>0.2)&(p<0.8)).mean():.1f}%")

# ---- D. how many clips would we extract, and how big
tot_png = sum(counts[k] for k, *_ in pos)
print(f"\nD. all {len(pos)} usable accident clips = {tot_png:,} png files in images/")


1962 rows   accident-occurred: {1: 1945, 0: 17}

archive: 1962 clips with images/

A. sheet rows per type vs archive clips per type:
   type   1: sheet   53 rows (video 1..53)   archive   53 clips
   type   2: sheet    6 rows (video 1..6)   archive    6 clips
   type   3: sheet   17 rows (video 1..17)   archive   17 clips
   type   4: sheet   13 rows (video 1..13)   archive   13 clips
   type   5: sheet  157 rows (video 1..157)   archive  157 clips
   type   6: sheet  119 rows (video 1..119)   archive  119 clips
   type   7: sheet   12 rows (video 1..12)   archive   12 clips
   type   8: sheet   57 rows (video 1..57)   archive   57 clips
   -> counts agree for 52/52 type folders

B. sheet 'total frames' vs actual PNG count per clip:
   mapped 1962/1962 rows to a folder
   EXACT frame-count match: 1949/1962
   worst disagreements (type,video): archive_png vs sheet_total
      (48, 65)  330 vs 192   diff +138
      (5, 40)  382 vs 482   diff -100
      (43, 80)  338 vs 388   diff -50
   

In [45]:
# === CELL A: choose 220 clips, then PROVE the pipeline on ONE ===
import collections, glob, os, subprocess, time
import pandas as pd, numpy as np

FPS   = 30          # assembly rate. Position is a RATIO so this cancels in gate 3a; it only
                    # has to be the SAME number used for the annotation seconds below.
N_PICK = 220
SEED   = 0
WORK, VID = "/content/work", "/content/dada_mp4"
os.makedirs(VID, exist_ok=True)

df = pd.read_excel(glob.glob(os.path.join(SRC, "*.xlsx"))[0], sheet_name="Sheet1")
df.columns = [str(c) for c in df.columns]
ACC = [c for c in df.columns if "accident frame" in c][0]
TOT = [c for c in df.columns if "total frames" in c][0]
OCC = [c for c in df.columns if "accident occurred" in c][0]

counts = collections.defaultdict(int)
for l in open("/content/listing.txt"):
    f = l.split(None, 5)
    if len(f) != 6:
        continue
    q = f[5].replace("\\", "/").strip().split("/")
    if len(q) == 5 and q[0] == "DADA2000" and q[3] == "images" and q[4].lower().endswith(".png"):
        counts[(int(q[1]), int(q[2]))] += 1

# Keep only rows we can trust: an accident happened, the frame count on disk matches the
# sheet exactly, and the accident frame lies inside the clip. A row failing any of these
# is dropped, never repaired -- a repaired annotation is an invented one.
rows = []
for _, r in df.iterrows():
    k = (int(r["type"]), int(r["video"]))
    if k not in counts or r[OCC] != 1:
        continue
    if counts[k] != int(r[TOT]):
        continue
    a, t = int(r[ACC]), int(r[TOT])
    if not (0 < a <= t):
        continue
    rows.append({"key": k, "id": f"{k[0]}_{k[1]:03d}", "acc": a, "tot": t, "pos": a / t})
print(f"usable clips: {len(rows)} of {len(df)} sheet rows")

# Stratify on COLLISION POSITION -- the annotated quantity, never on any score (D41).
# Even deciles keep the spread that makes the correlation estimable.
rng = np.random.default_rng(SEED)
pos = np.array([r["pos"] for r in rows])
edges = np.quantile(pos, np.linspace(0, 1, 11))
pick = []
for i in range(10):
    lo, hi = edges[i], edges[i + 1]
    band = [r for r in rows if (lo <= r["pos"] < hi) or (i == 9 and r["pos"] == hi)]
    take = min(N_PICK // 10, len(band))
    pick += [band[j] for j in rng.choice(len(band), take, replace=False)]
pick.sort(key=lambda r: r["id"])
p = np.array([r["pos"] for r in pick])
print(f"picked {len(pick)}  position median {np.median(p):.3f}  "
      f"IQR {np.percentile(p,75)-np.percentile(p,25):.3f}  range {p.min():.3f}-{p.max():.3f}")
print(f"  in final 10%: {100*(p>0.9).mean():.1f}%   mid-clip: {100*((p>0.2)&(p<0.8)).mean():.1f}%")
print(f"  frames total: {sum(r['tot'] for r in pick):,}")

def build(rec, quiet=True):
    """Extract one clip's RGB frames, stitch to mp4, delete the frames."""
    t, v = rec["key"]
    src = f"DADA2000/{t}/{v:03d}/images"
    out = os.path.join(VID, f"{rec['id']}.mp4")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return out, 0.0, True
    t0 = time.time()
    subprocess.run(["7z", "x", LAST, f"-o{WORK}", f"{src}/*", "-y"],
                   capture_output=True, text=True, timeout=3600, check=True)
    d = os.path.join(WORK, src)
    pngs = sorted(glob.glob(os.path.join(d, "*.png")))
    assert len(pngs) == rec["tot"], f"{rec['id']}: {len(pngs)} png vs sheet {rec['tot']}"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-framerate", str(FPS),
                    "-pattern_type", "glob", "-i", os.path.join(d, "*.png"),
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", out],
                   capture_output=True, text=True, timeout=3600, check=True)
    subprocess.run(["rm", "-rf", os.path.join(WORK, f"DADA2000/{t}/{v:03d}")], check=False)
    return out, time.time() - t0, False

# ---- PROVE IT ON ONE ----
r0 = pick[0]
print(f"\nbuilding {r0['id']}  ({r0['tot']} frames, collision at frame {r0['acc']} "
      f"= position {r0['pos']:.3f})")
out, dt, _ = build(r0)
print(f"  {dt:.1f}s   {os.path.getsize(out)/2**20:.1f} MiB")

import cv2
cap = cv2.VideoCapture(out)
ok, _ = cap.read()
n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f"  decodes {ok}   frames {n:.0f}   fps {fps:.2f}   duration {n/max(fps,1):.2f}s")
assert ok, "mp4 will not decode -- stop"
assert abs(n - r0["tot"]) <= 1, f"frame count changed in assembly: {n} vs {r0['tot']}"
print("  ✅ frame count preserved end to end -- the time base is intact")
print(f"\nestimate for {len(pick)} clips: {dt*len(pick)/60:.0f} min "
      f"(~{dt*len(pick)/3600:.1f} h)")
if dt * len(pick) / 3600 > 4:
    print("  ⚠️ STOP CONDITION: > 4 h. Report back before running cell B.")


usable clips: 1932 of 1962 sheet rows
picked 220  position median 0.541  IQR 0.282  range 0.084-0.995
  in final 10%: 3.6%   mid-clip: 85.0%
  frames total: 73,200

building 10_004  (280 frames, collision at frame 170 = position 0.607)
  0.0s   3.4 MiB
  decodes True   frames 280   fps 30.00   duration 9.33s
  ✅ frame count preserved end to end -- the time base is intact

estimate for 220 clips: 0 min (~0.0 h)


In [47]:
# === DEBUG: why did ffmpeg fail? Frames are already extracted, so this is quick. ===
import glob, os, subprocess, collections

d = "/content/work/DADA2000/10/004/images"
pngs = sorted(glob.glob(os.path.join(d, "*.png")))
print(f"{len(pngs)} png files")
print("first:", os.path.basename(pngs[0]), " last:", os.path.basename(pngs[-1]))

# 1. image dimensions -- libx264 + yuv420p REQUIRES even width and height
from PIL import Image
dims = collections.Counter()
for p in pngs[::max(1, len(pngs)//40)]:
    dims[Image.open(p).size] += 1
print("\nsizes seen (sampled):", dims.most_common())
for (w, h), n in dims.items():
    print(f"   {w}x{h}  even? width={w%2==0} height={h%2==0}")
if len(dims) > 1:
    print("   ⚠️ frames are NOT all the same size -- needs a scale filter regardless")

# 2. is libx264 even in this ffmpeg build?
enc = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True)
for name in ("libx264", "libopenh264", "mpeg4", "libvpx-vp9"):
    print(f"   encoder {name:12s} {'present' if name in enc.stdout else 'MISSING'}")

# 3. the real error, unsuppressed
print("\n--- ffmpeg stderr (original command) ---")
r = subprocess.run(["ffmpeg", "-y", "-framerate", "30", "-pattern_type", "glob",
                    "-i", os.path.join(d, "*.png"), "-c:v", "libx264",
                    "-pix_fmt", "yuv420p", "-crf", "18", "/content/t1.mp4"],
                   capture_output=True, text=True)
print("exit", r.returncode)
print(r.stderr[-2500:])

# 4. the likely fix: force even dimensions with a crop (loses at most 1px per axis)
print("\n--- retry with even-dimension crop ---")
r2 = subprocess.run(["ffmpeg", "-y", "-framerate", "30", "-pattern_type", "glob",
                     "-i", os.path.join(d, "*.png"),
                     "-vf", "crop=trunc(iw/2)*2:trunc(ih/2)*2",
                     "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
                     "/content/t2.mp4"], capture_output=True, text=True)
print("exit", r2.returncode)
if r2.returncode != 0:
    print(r2.stderr[-2500:])
else:
    import cv2
    cap = cv2.VideoCapture("/content/t2.mp4")
    ok, _ = cap.read()
    n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS)
    w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    print(f"   ✅ {os.path.getsize('/content/t2.mp4')/2**20:.1f} MiB  decodes {ok}  "
          f"frames {n:.0f} (want {len(pngs)})  fps {fps:.2f}  {w:.0f}x{h:.0f}")


0 png files


IndexError: list index out of range

## 3 — What is actually in there: video, or image frames?🔴 **The gating check.** The HuggingFace mirror of DADA stores clips as directories of PNG frames.Our scorer decodes **video**. If this archive is frames too, the plan changes and this notebookstops here.

In [ ]:
import collections, os

# 7z -ba output: "date time attr size compressed name" -- name is the remainder after 5 fields.
paths = []
for l in lines:
    f = l.split(None, 5)
    if len(f) == 6:
        paths.append(f[5].replace("\\", "/"))

ext = collections.Counter(os.path.splitext(p)[1].lower() for p in paths)
print("extensions:", ext.most_common(10))

n_video = sum(v for k, v in ext.items() if k in (".mp4", ".avi", ".mov", ".mkv"))
n_image = sum(v for k, v in ext.items() if k in (".png", ".jpg", ".jpeg"))
print(f"\nvideo files: {n_video}    image files: {n_image}")

print("\nsample paths:")
for p in paths[:10]:
    print("  ", p)

if n_video == 0 and n_image > 0:
    raise SystemExit(
        "STOP. This archive holds image frames, not video. Our scorer decodes video, so this is a "
        "different job -- report back before going further. Do not improvise a frame loader here.")

## 4 — Match the 221 clips we needIds come from `dada2000_small_test_concensus.csv`, embedded below so this notebook needs no networkand cannot drift from the repo copy. Each is `{type}_{clip}.mp4` → the archive most likely stores itas `{type}/{clip}...`, so we match on both shapes and **report coverage honestly** rather thansilently taking whatever matched.

In [ ]:
WANTED = ["10_068.mp4","10_069.mp4","10_070.mp4","10_071.mp4","10_072.mp4","10_073.mp4","10_074.mp4","10_075.mp4","10_076.mp4","10_077.mp4","10_078.mp4","10_079.mp4","10_080.mp4","10_081.mp4","10_082.mp4","10_083.mp4","10_084.mp4","11_097.mp4","11_098.mp4","11_099.mp4","11_100.mp4","11_101.mp4","11_102.mp4","11_103.mp4","11_104.mp4","11_105.mp4","11_106.mp4","11_107.mp4","11_108.mp4","11_109.mp4","11_110.mp4","11_111.mp4","11_112.mp4","11_113.mp4","11_114.mp4","11_115.mp4","11_116.mp4","11_117.mp4","11_118.mp4","11_119.mp4","11_120.mp4","12_015.mp4","12_016.mp4","12_017.mp4","13_004.mp4","14_013.mp4","14_014.mp4","14_015.mp4","15_001.mp4","16_001.mp4","17_001.mp4","18_004.mp4","19_001.mp4","1_022.mp4","1_023.mp4","1_024.mp4","1_025.mp4","1_026.mp4","1_027.mp4","20_001.mp4","21_001.mp4","22_007.mp4","22_008.mp4","23_001.mp4","24_001.mp4","25_001.mp4","26_001.mp4","27_006.mp4","27_007.mp4","28_035.mp4","28_036.mp4","28_037.mp4","28_038.mp4","28_039.mp4","28_041.mp4","28_042.mp4","28_043.mp4","29_029.mp4","29_030.mp4","29_031.mp4","29_032.mp4","29_033.mp4","29_034.mp4","29_035.mp4","2_004.mp4","30_014.mp4","30_015.mp4","30_016.mp4","31_007.mp4","31_008.mp4","32_007.mp4","32_008.mp4","33_011.mp4","33_012.mp4","33_013.mp4","34_088.mp4","34_089.mp4","34_090.mp4","34_091.mp4","34_092.mp4","34_093.mp4","34_094.mp4","34_095.mp4","34_096.mp4","34_097.mp4","34_098.mp4","34_099.mp4","34_100.mp4","34_101.mp4","34_102.mp4","34_103.mp4","34_104.mp4","34_105.mp4","34_106.mp4","34_107.mp4","34_108.mp4","34_109.mp4","35_005.mp4","36_006.mp4","37_001.mp4","38_034.mp4","38_035.mp4","38_036.mp4","38_037.mp4","38_038.mp4","38_039.mp4","38_040.mp4","38_041.mp4","39_020.mp4","39_021.mp4","39_022.mp4","39_023.mp4","39_024.mp4","3_006.mp4","3_007.mp4","40_080.mp4","40_081.mp4","40_082.mp4","40_083.mp4","40_084.mp4","40_085.mp4","40_086.mp4","40_087.mp4","40_088.mp4","40_089.mp4","40_090.mp4","40_091.mp4","40_092.mp4","40_093.mp4","40_094.mp4","40_095.mp4","40_096.mp4","40_097.mp4","40_098.mp4","40_099.mp4","41_007.mp4","41_008.mp4","42_004.mp4","43_006.mp4","43_007.mp4","44_002.mp4","45_002.mp4","46_016.mp4","46_017.mp4","46_018.mp4","46_019.mp4","47_020.mp4","47_021.mp4","47_022.mp4","47_023.mp4","47_024.mp4","48_002.mp4","49_008.mp4","49_009.mp4","4_007.mp4","4_008.mp4","50_002.mp4","51_009.mp4","51_010.mp4","52_001.mp4","53_001.mp4","54_017.mp4","54_018.mp4","54_019.mp4","54_020.mp4","5_064.mp4","5_065.mp4","5_066.mp4","5_067.mp4","5_068.mp4","5_069.mp4","5_070.mp4","5_071.mp4","5_072.mp4","5_073.mp4","5_074.mp4","5_075.mp4","5_076.mp4","5_077.mp4","5_078.mp4","5_079.mp4","6_049.mp4","6_050.mp4","6_051.mp4","6_052.mp4","6_053.mp4","6_054.mp4","6_055.mp4","6_056.mp4","6_057.mp4","6_058.mp4","6_059.mp4","6_060.mp4","7_006.mp4","8_021.mp4","8_022.mp4","8_023.mp4","8_024.mp4","8_025.mp4","9_009.mp4","9_010.mp4"]
print(len(WANTED), "ids wanted")

import re, os

def keys(name):
    """{type}_{clip} in both the flat and the nested spelling."""
    stem = os.path.splitext(name)[0]
    t, c = stem.split("_", 1)
    return stem, f"{t}/{c}"

matched, missing = {}, []
lower = {p.lower(): p for p in paths}

for w in WANTED:
    flat, nested = keys(w)
    hit = [p for p in paths if p.lower().endswith(f"/{flat.lower()}.mp4")
           or p.lower().endswith(f"/{nested.lower()}.mp4")
           or f"/{nested.lower()}/" in p.lower()
           or p.lower() == f"{flat.lower()}.mp4"]
    if hit:
        matched[w] = sorted(hit)
    else:
        missing.append(w)

print(f"matched {len(matched)}/{len(WANTED)}   missing {len(missing)}")
if missing:
    print("\nfirst missing:", missing[:10])
    print("\nIf most are missing, the archive uses a different layout than assumed. Print a few")
    print("real paths above and report back -- do NOT loosen the matcher until it 'finds' things.")

for w in list(matched)[:5]:
    print(f"  {w}  ->  {matched[w]}")

## 5 — Prove it on ONE clip firstExtract a single clip, check it decodes, and **time it**. That measured time is what we extrapolatefrom — the same rule the repo applies to scoring runs: measure, never assert.**Stop condition:** if one clip implies more than ~4 hours for 221, stop and report rather thanstarting a run that will not finish.

In [ ]:
import subprocess, time, os

OUT = "/content/dada_out"
os.makedirs(OUT, exist_ok=True)

first = sorted(matched)[0]
member = matched[first][0]
print("extracting:", member)

t0 = time.time()
r = subprocess.run(["7z", "e", LAST, f"-o{OUT}", member, "-y"],
                   capture_output=True, text=True, timeout=7200)
dt = time.time() - t0
print(f"exit {r.returncode}   {dt:.1f}s")
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit("extraction failed -- report this, do not retry blindly")

got = os.listdir(OUT)
print("wrote:", got)
sz = os.path.getsize(os.path.join(OUT, got[0])) / 2**20
print(f"size: {sz:.1f} MiB")

# Does it actually decode? A file of the right size that will not open is worse than no file.
!pip -q install opencv-python-headless 2>/dev/null
import cv2
cap = cv2.VideoCapture(os.path.join(OUT, got[0]))
ok, frame = cap.read()
n = cap.get(cv2.CAP_PROP_FRAME_COUNT); fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f"decodes: {ok}   frames: {n:.0f}   fps: {fps:.2f}   duration: {n/max(fps,1):.2f}s")
assert ok, "file extracted but will not decode -- stop"

print(f"\nestimate for 221 clips: {dt*221/60:.0f} min   (~{dt*221/3600:.1f} h)")
if dt * 221 / 3600 > 4:
    print("\nSTOP CONDITION: > 4 h implied. Report back before running the full extraction.")

## 6 — Extract all 221, then copy to DriveWritten to Colab's local disk first (fast), then copied to Drive in one pass. Resumable: re-runningskips clips already present, so a disconnect costs only the remaining clips.

In [ ]:
import shutil, time, os

members = [matched[w][0] for w in sorted(matched)]
done = set(os.listdir(OUT))
todo = [m for m in members if os.path.basename(m) not in done]
print(f"{len(members)} wanted, {len(done)} already local, {len(todo)} to extract")

t0 = time.time()
CHUNK = 20
for i in range(0, len(todo), CHUNK):
    batch = todo[i:i + CHUNK]
    r = subprocess.run(["7z", "e", LAST, f"-o{OUT}", *batch, "-y"],
                       capture_output=True, text=True, timeout=14400)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit(f"failed at batch starting {i} -- re-run this cell to resume")
    el = time.time() - t0
    n = min(i + CHUNK, len(todo))
    print(f"  {n}/{len(todo)}   {el/60:.1f} min   eta {el/n*(len(todo)-n)/60:.1f} min", flush=True)

files = sorted(f for f in os.listdir(OUT) if f.lower().endswith(".mp4"))
total = sum(os.path.getsize(os.path.join(OUT, f)) for f in files)
print(f"\n{len(files)} clips, {total/2**30:.2f} GiB")

## 7 — Manifest, then copy to DriveThe manifest carries a SHA-256 per clip so the Mac can prove the download is byte-identical to whatwas extracted here. A truncated clip that still decodes would corrupt gate 3a silently.

In [ ]:
import hashlib, json, os, shutil

DEST = "/content/drive/MyDrive/dada2000_test221"
os.makedirs(DEST, exist_ok=True)

manifest = {}
for f in files:
    p = os.path.join(OUT, f)
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for blk in iter(lambda: fh.read(1 << 20), b""):
            h.update(blk)
    manifest[f] = {"sha256": h.hexdigest(), "bytes": os.path.getsize(p)}

with open(os.path.join(OUT, "manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)

missing_note = {"wanted": len(WANTED), "matched": len(matched), "missing": missing}
with open(os.path.join(OUT, "coverage.json"), "w") as fh:
    json.dump(missing_note, fh, indent=2)

for f in os.listdir(OUT):
    dst = os.path.join(DEST, f)
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(os.path.join(OUT, f)):
        shutil.copy2(os.path.join(OUT, f), dst)

print(f"copied {len(os.listdir(DEST))} files to {DEST}")
print(f"coverage: {len(matched)}/{len(WANTED)} clips")
print("\nNext: download that Drive folder to the Mac, then run scripts/score_external.py")

In [52]:
# === FIX: the "*.png" files are actually JPEG. Detect, then re-run the one-clip proof. ===
import collections, glob, os, subprocess, time

MAGIC = {b"\x89PNG": "png", b"\xff\xd8\xff": "mjpeg"}

def sniff(path):
    with open(path, "rb") as f:
        head = f.read(4)
    for m, dec in MAGIC.items():
        if head.startswith(m):
            return dec
    return None

d = "/content/work/DADA2000/10/004/images"
pngs = sorted(glob.glob(os.path.join(d, "*.png")))
seen = collections.Counter(sniff(p) for p in pngs)          # EVERY file, not a sample
print("actual formats in this clip:", dict(seen))
assert None not in seen, "some files are neither PNG nor JPEG -- stop"
assert len(seen) == 1, f"MIXED formats in one clip {dict(seen)} -- ffmpeg needs one decoder"
DEC = next(iter(seen))
print(f"decoder to use: {DEC}\n")

def build(rec, vid_dir="/content/dada_mp4", work="/content/work", fps=30):
    """Extract one clip's frames, stitch to mp4 at `fps`, delete the frames.

    The frames are JPEG despite the .png extension, so the decoder is named explicitly
    (-c:v before -i). Letting ffmpeg infer it from the extension is what failed.
    The crop forces even dimensions, which libx264+yuv420p requires; it is a no-op at
    1584x660 and a 1px trim on any odd-sized clip.
    """
    t, v = rec["key"]
    src = f"DADA2000/{t}/{v:03d}/images"
    out = os.path.join(vid_dir, f"{rec['id']}.mp4")
    os.makedirs(vid_dir, exist_ok=True)
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return out, 0.0, True
    t0 = time.time()
    subprocess.run(["7z", "x", LAST, f"-o{work}", f"{src}/*", "-y"],
                   capture_output=True, text=True, timeout=3600, check=True)
    dd = os.path.join(work, src)
    frames = sorted(glob.glob(os.path.join(dd, "*.png")))
    assert len(frames) == rec["tot"], f"{rec['id']}: {len(frames)} files vs sheet {rec['tot']}"
    dec = sniff(frames[0])
    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error",
                        "-framerate", str(fps), "-f", "image2", "-c:v", dec,
                        "-pattern_type", "glob", "-i", os.path.join(dd, "*.png"),
                        "-vf", "crop=trunc(iw/2)*2:trunc(ih/2)*2",
                        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", out],
                       capture_output=True, text=True, timeout=3600)
    if r.returncode != 0:
        raise RuntimeError(f"{rec['id']} ffmpeg failed:\n{r.stderr[-1500:]}")
    subprocess.run(["rm", "-rf", os.path.join(work, f"DADA2000/{t}/{v:03d}")], check=False)
    return out, time.time() - t0, False

# ---- prove it on the same clip ----
r0 = pick[0]
print(f"building {r0['id']}  ({r0['tot']} frames, collision at frame {r0['acc']} "
      f"= position {r0['pos']:.3f})")
out, dt, _ = build(r0)
print(f"  {dt:.1f}s   {os.path.getsize(out)/2**20:.1f} MiB")

import cv2
cap = cv2.VideoCapture(out)
ok, _ = cap.read()
n, fps_out = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS)
w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
cap.release()
print(f"  decodes {ok}   frames {n:.0f} (want {r0['tot']})   fps {fps_out:.2f}   {w:.0f}x{h:.0f}")
assert ok, "mp4 will not decode -- stop"
assert abs(n - r0["tot"]) <= 1, f"frame count changed in assembly: {n} vs {r0['tot']}"
print("  ✅ frame count preserved end to end -- the time base is intact")
print(f"\nestimate for {len(pick)} clips: {dt*len(pick)/60:.0f} min (~{dt*len(pick)/3600:.1f} h)")
if dt * len(pick) / 3600 > 4:
    print("  ⚠️ STOP CONDITION: > 4 h. Report back before the full run.")


actual formats in this clip: {}


AssertionError: MIXED formats in one clip {} -- ffmpeg needs one decoder

In [48]:
# === CELL B: build all 220 clips, write the annotation, copy to Drive ===
import csv, hashlib, json, os, shutil, time

FPS  = 30
VID  = "/content/dada_mp4"
DEST = "/content/drive/MyDrive/dada2000_gate3a"
os.makedirs(DEST, exist_ok=True)

t0, failed = time.time(), []
for n, rec in enumerate(pick, 1):
    try:
        out, dt, cached = build(rec)
    except Exception as e:
        failed.append((rec["id"], str(e)[:200]))
        print(f"  {n}/{len(pick)}  {rec['id']}  FAILED: {str(e)[:120]}", flush=True)
        continue
    if n % 10 == 0 or n == len(pick):
        el = time.time() - t0
        print(f"  {n}/{len(pick)}  {el/60:.1f} min elapsed, "
              f"eta {el/n*(len(pick)-n)/60:.1f} min", flush=True)

built = sorted(f for f in os.listdir(VID) if f.endswith(".mp4"))
print(f"\nbuilt {len(built)}/{len(pick)}   failed {len(failed)}")
for i, m in failed[:10]:
    print("   ", i, m)

# ---- annotation, in the exact shape eval/gate3_mechanism.py reads ----
# Time-of-collision is accident_frame / FPS, using the SAME FPS the video was stitched at,
# so seconds and frames cannot drift apart. Position (a ratio) is carried too and is
# frame-rate-independent -- it is the quantity the gate actually turns on.
by_id = {r["id"]: r for r in pick}
rows = []
for f in built:
    r = by_id[os.path.splitext(f)[0]]
    rows.append({"id": f,
                 "Event-type": "Collision",
                 "Time-of-collision": round(r["acc"] / FPS, 4),
                 "accident_frame": r["acc"],
                 "total_frames": r["tot"],
                 "position": round(r["pos"], 6),
                 "clip_seconds": round(r["tot"] / FPS, 4)})
ann = os.path.join(VID, "dada_gate3a_annotation.csv")
with open(ann, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0]))
    w.writeheader(); w.writerows(rows)
print(f"\nannotation: {len(rows)} rows -> {os.path.basename(ann)}")

import numpy as np
p = np.array([r["position"] for r in rows])
s = np.array([r["Time-of-collision"] for r in rows])
print(f"  position  median {np.median(p):.3f}  IQR {np.percentile(p,75)-np.percentile(p,25):.3f}"
      f"  range {p.min():.3f}-{p.max():.3f}")
print(f"  seconds   median {np.median(s):.2f}s  IQR {np.percentile(s,75)-np.percentile(s,25):.2f}s")
print(f"  in final 10%: {100*(p>0.9).mean():.1f}%   (Nexar positives 73.7%)")

# ---- manifest: sha256 per clip, so the Mac can prove the download is byte-identical ----
man = {}
for f in built:
    h = hashlib.sha256()
    with open(os.path.join(VID, f), "rb") as fh:
        for blk in iter(lambda: fh.read(1 << 20), b""):
            h.update(blk)
    man[f] = {"sha256": h.hexdigest(), "bytes": os.path.getsize(os.path.join(VID, f))}
with open(os.path.join(VID, "manifest.json"), "w") as fh:
    json.dump({"fps": FPS, "n": len(man), "failed": failed, "clips": man}, fh, indent=2)

# ---- copy to Drive ----
for f in os.listdir(VID):
    src, dst = os.path.join(VID, f), os.path.join(DEST, f)
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        shutil.copy2(src, dst)
tot = sum(os.path.getsize(os.path.join(DEST, f)) for f in os.listdir(DEST))
print(f"\ncopied {len(os.listdir(DEST))} files to {DEST}  ({tot/2**20:.0f} MiB)")
print("Next: download that folder to the Mac, verify manifest.json, then score.")


  10/220  0.0 min elapsed, eta 0.0 min
  20/220  0.0 min elapsed, eta 0.0 min
  30/220  0.0 min elapsed, eta 0.0 min
  40/220  0.0 min elapsed, eta 0.0 min
  50/220  0.0 min elapsed, eta 0.0 min
  60/220  0.0 min elapsed, eta 0.0 min
  70/220  0.0 min elapsed, eta 0.0 min
  80/220  0.0 min elapsed, eta 0.0 min
  90/220  0.0 min elapsed, eta 0.0 min
  100/220  0.0 min elapsed, eta 0.0 min
  110/220  0.0 min elapsed, eta 0.0 min
  120/220  0.0 min elapsed, eta 0.0 min
  130/220  0.0 min elapsed, eta 0.0 min
  140/220  0.0 min elapsed, eta 0.0 min
  150/220  0.0 min elapsed, eta 0.0 min


KeyboardInterrupt: 

In [51]:
# === RESUME: remount Drive, clean the interrupted clip, report what is left ===
import glob, os, shutil, subprocess, time

VID, WORK = "/content/dada_mp4", "/content/work"

# 1. after a disconnect the Drive FUSE mount goes stale and reads hang forever.
#    Remount before touching the archive.
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# 2. prove the archive is actually readable again, with a timeout so a stale mount
#    fails fast instead of hanging the cell like last time.
t0 = time.time()
try:
    sz = os.path.getsize(LAST)
    with open(LAST, "rb") as f:
        f.read(1 << 20)
    print(f"archive readable: {os.path.basename(LAST)} {sz/2**30:.2f} GiB  "
          f"({time.time()-t0:.1f}s)")
except Exception as e:
    raise SystemExit(f"archive still not readable ({e}) -- re-run this cell")

# 3. the clip that was interrupted mid-extraction leaves a partial folder. 7z -y would
#    overwrite it, but a half-written folder could also pass an unlucky count check, so
#    remove anything left in the work dir rather than trusting it.
if os.path.isdir(WORK):
    n = len(glob.glob(os.path.join(WORK, "DADA2000/*/*")))
    shutil.rmtree(WORK, ignore_errors=True)
    print(f"cleared work dir ({n} partial clip folders)")

# 4. what is done, what is left
done = {os.path.splitext(f)[0] for f in os.listdir(VID)
        if f.endswith(".mp4") and os.path.getsize(os.path.join(VID, f)) > 0}
todo = [r for r in pick if r["id"] not in done]
print(f"\nbuilt {len(done)}/{len(pick)}   remaining {len(todo)}")
print(f"estimated {len(todo)*24/60:.0f} min at 24 s/clip")

# 5. spot-check a finished clip -- a file can exist and still be truncated
import cv2
if done:
    s = sorted(done)[0]
    by_id = {r["id"]: r for r in pick}
    cap = cv2.VideoCapture(os.path.join(VID, f"{s}.mp4"))
    ok, _ = cap.read(); n = cap.get(cv2.CAP_PROP_FRAME_COUNT); cap.release()
    print(f"spot-check {s}: decodes {ok}, frames {n:.0f} (want {by_id[s]['tot']})")
    assert ok and abs(n - by_id[s]["tot"]) <= 1, "a finished clip is bad -- tell Claude"
    print("ok  finished clips are intact")

print("\n-> now re-run CELL B. It skips the", len(done), "already built.")


Mounted at /content/drive
archive readable: DADA2000.zip 17.25 GiB  (0.0s)

built 153/220   remaining 67
estimated 27 min at 24 s/clip
spot-check 10_004: decodes True, frames 280 (want 280)
ok  finished clips are intact

-> now re-run CELL B. It skips the 153 already built.
